# ExamScheduling — Colab Reproduction Runner

Runs the full benchmark batch for CSC 2400: **13 algorithms × 8 ITC 2007 instances × 3 seeds**. Expect ~60–120 minutes on a free-tier Colab CPU runtime; a Pro runtime halves that.

**What this notebook does:**
1. Installs build deps (g++, OR-Tools, pandas, matplotlib, pytest)
2. Clones the repo + builds the headers-only C++ solver
3. Smoke-tests one algorithm × one instance to catch env breakage early
4. Loops every algorithm × every ITC 2007 set × `NUM_SEEDS` seeds, saving per-run JSON into `results/colab_batch/`
5. Aggregates results into a single CSV + regenerates the paper figures
6. Zips `results/` and triggers a browser download

> **If a step fails mid-batch:** the outer loop `continue`s past failures and still writes whatever completed. Check `results/colab_batch/failures.log` afterwards.

## 1. Environment setup

In [ ]:
!apt-get install -y build-essential > /dev/null 2>&1
!pip install -q ortools pulp numpy pandas matplotlib plotly pytest

## 1b. (optional) Mount Google Drive

Run this cell **before** the zip/download cells to auto-backup the result zips to your Drive. Skip it if you only want the browser download.

> `files.download(...)` by itself sends the zip to your computer's Downloads folder — it doesn't touch Drive. Mounting here gives you both: the Drive copy survives runtime disconnects, and the browser download is still triggered.

In [ ]:
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_BACKUP_DIR = '/content/drive/MyDrive/exam_scheduling_batches'
    os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)
    os.environ['DRIVE_BACKUP_DIR'] = DRIVE_BACKUP_DIR
    print(f'Drive mounted. Zips will also land in: {DRIVE_BACKUP_DIR}')
except (ImportError, ModuleNotFoundError):
    print('Not running on Colab — skipping Drive mount.')

## 2. Clone the repo

Update `GH_USER` / `REPO` if you forked. The cell is idempotent — re-running after the clone just `cd`s into the checkout.

In [ ]:
import os

GH_USER = 'Dialovos'
REPO = 'exam-scheduling-algos-analyze'

if not os.path.exists(REPO):
    !git clone --depth 1 https://github.com/{GH_USER}/{REPO}.git
%cd {REPO}
!pwd && git log -1 --oneline

## 3. Build the C++ solver + sanity smoke

In [ ]:
!make
print('\n--- smoke: Tabu on set1, seed 42 ---')
!python main.py --dataset instances/exam_comp_set1.exam --algo tabu --seed 42 --no-batch --quiet

## 4. Full batch

Drops results into `results/colab_batch/<algo>_<set>_<seed>/`. Adjust `ALGOS` / `DATASETS` / `NUM_SEEDS` below to trim scope — the default is the full paper matrix.

CP-SAT (`cpsat`) has a 60 s time limit per run; the heuristics finish in seconds. IP (`ip`) is skipped for `n > 500` exams automatically inside `main.py`.

In [ ]:
import glob, os, subprocess, time
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed

ALGOS = ['greedy', 'tabu', 'kempe', 'sa', 'alns', 'gd', 'abc', 'ga',
         'lahc', 'woa', 'hho', 'cpsat', 'vns']
DATASETS = sorted(glob.glob('instances/exam_comp_set*.exam'))
NUM_SEEDS = 3

# Worker count: every heuristic is single-threaded C++, so pack runs across cores.
# Leave one vCPU free for the Python driver + I/O; CP-SAT inside the cpsat algo
# is internally single-worker here (num_workers=8 is only set by the Python IP path).
MAX_WORKERS = max(1, (os.cpu_count() or 2) - 1)

BATCH_ROOT = Path('results/colab_batch')
BATCH_ROOT.mkdir(parents=True, exist_ok=True)
FAIL_LOG = BATCH_ROOT / 'failures.log'
if FAIL_LOG.exists(): FAIL_LOG.unlink()


def run_one(algo, dataset, seed):
    """Child-process entry: run one (algo, dataset, seed) combination."""
    ds_name = Path(dataset).stem
    run_dir = BATCH_ROOT / f'{algo}_{ds_name}_seed{seed}'
    run_dir.mkdir(exist_ok=True)
    cmd = ['python', 'main.py', '--dataset', dataset, '--algo', algo,
           '--seed', str(seed), '--no-batch', '--quiet',
           '--output', str(run_dir)]
    try:
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=600)
        if r.returncode != 0:
            return (algo, ds_name, seed, 'fail', r.stderr[-500:])
        return (algo, ds_name, seed, 'ok', '')
    except subprocess.TimeoutExpired:
        return (algo, ds_name, seed, 'timeout', '600s')


jobs = [(a, d, s) for a in ALGOS for d in DATASETS
        for s in range(42, 42 + NUM_SEEDS)]
total = len(jobs)
print(f'Launching {total} runs across {MAX_WORKERS} workers '
      f'({len(ALGOS)} algos × {len(DATASETS)} sets × {NUM_SEEDS} seeds)')
print(f'Output: {BATCH_ROOT.resolve()}')

started = time.time()
completed = 0
with ProcessPoolExecutor(max_workers=MAX_WORKERS) as pool:
    futures = {pool.submit(run_one, *j): j for j in jobs}
    for fut in as_completed(futures):
        algo, ds_name, seed, status, msg = fut.result()
        completed += 1
        elapsed = time.time() - started
        eta = elapsed / completed * (total - completed)
        tag = '  ok' if status == 'ok' else f' {status.upper()}'
        print(f'  [{completed:>3}/{total}] {algo:<7} {ds_name:<18} seed={seed}'
              f'  elapsed={elapsed/60:5.1f}m  eta={eta/60:5.1f}m {tag}',
              flush=True)
        if status != 'ok':
            with open(FAIL_LOG, 'a') as f:
                f.write(f'{algo}/{ds_name}/seed{seed}: {status}\n{msg}\n\n')

print(f'\nDone in {(time.time()-started)/60:.1f} min. Failures:',
      FAIL_LOG.stat().st_size if FAIL_LOG.exists() else 0, 'bytes')

## 5. Aggregate + regenerate figures

Walks `results/colab_batch/`, assembles one long-form DataFrame, runs the full matplotlib plot suite into `results/colab_batch/figures/`.

In [ ]:
import json, pandas as pd
from pathlib import Path
from utils.plots import generate_all_plots

BATCH_ROOT = Path('results/colab_batch')
rows = []
for run_dir in sorted(BATCH_ROOT.iterdir()):
    if not run_dir.is_dir(): continue
    sb = run_dir / 'soft_breakdown.json'
    if not sb.exists(): continue
    # Run dirs look like `{algo}_{dataset}_seed{N}`; dataset can contain
    # underscores (e.g. `exam_comp_set1`), so peel the seed suffix first.
    head, _, seed_tag = run_dir.name.rpartition('_seed')
    if not seed_tag:
        continue  # not a standard run dir
    algo, _, ds_name = head.partition('_')
    try:
        seed = int(seed_tag)
    except ValueError:
        continue
    breakdown = json.loads(sb.read_text())
    for algo_label, parts in breakdown.items():
        row = {'algorithm': algo_label, 'dataset': ds_name, 'seed': seed,
               'soft_penalty': sum(parts.values())}
        row.update(parts)
        rows.append(row)

df = pd.DataFrame(rows)
print(f'Aggregated {len(df)} rows')
df.to_csv(BATCH_ROOT / 'aggregated.csv', index=False)

if not df.empty:
    # Minimum columns for plotting — synthesise runtime + feasibility from the CSV
    df['runtime'] = df.get('runtime', 0.0)
    df['feasible'] = df.get('feasible', True)
    generate_all_plots(df, output_dir=str(BATCH_ROOT / 'figures'))

## 6. Zip + download

Colab sandboxes are ephemeral — do this before the runtime disconnects.

In [ ]:
import os, shutil
from google.colab import files

!zip -r batch_colab.zip results/colab_batch/ > /dev/null

# If Drive was mounted earlier, auto-backup the zip there too.
drive_dir = os.environ.get('DRIVE_BACKUP_DIR', '')
if drive_dir and os.path.isdir(drive_dir):
    shutil.copy('batch_colab.zip', drive_dir)
    print(f'Drive copy: {drive_dir}/batch_colab.zip')

files.download('batch_colab.zip')

## 7. Post-export IP — CP-SAT with Tabu warm-start

> **Run this only after cell #6 finishes downloading `batch_colab.zip`.**
> If the runtime dies mid-IP, the main batch is already safe on disk + Drive.

This cell reuses the Tabu solutions from `results/colab_batch/` as a CP-SAT warm-start hint (`add_hint()`) — on feasible starting points that typically gives a 5–20× speedup versus cold CP-SAT. Runs **sequentially** per instance because CP-SAT internally parallelises across all vCPUs (`--ip-workers 0` = all cores) and outer parallelism would thrash.

- `IP_TIME` — wall-clock per instance (5 minutes is a reasonable default; raise it on A100 runtime if you want tighter bounds)
- `IP_MAX_EXAMS` — instances larger than this are skipped (default 900 covers 6 of 8 ITC sets)
- Results land in `results/colab_batch_ip/ip_<setname>/` and are zipped separately as `batch_colab_ip.zip`.

In [ ]:
import glob, subprocess, time
from pathlib import Path

IP_TIME = 300          # seconds per instance
IP_MAX_EXAMS = 900     # skip instances larger than this

main_batch = Path('results/colab_batch')
ip_batch = Path('results/colab_batch_ip')
ip_batch.mkdir(parents=True, exist_ok=True)
ip_fail = ip_batch / 'failures.log'
if ip_fail.exists(): ip_fail.unlink()

datasets = sorted(glob.glob('instances/exam_comp_set*.exam'))
t0 = time.time()

for ds in datasets:
    ds_name = Path(ds).stem
    # Pick the first available Tabu solution as warm-start (any seed works)
    warm_candidates = sorted(main_batch.glob(
        f'tabu_{ds_name}_seed*/solutions/solution_tabu_search_*.sln'))
    warm = str(warm_candidates[0]) if warm_candidates else None

    run_dir = ip_batch / f'ip_{ds_name}'
    run_dir.mkdir(exist_ok=True)
    cmd = ['python', 'main.py', '--dataset', ds, '--algo', 'ip',
           '--ip-time', str(IP_TIME),
           '--ip-max-exams', str(IP_MAX_EXAMS),
           '--ip-workers', '0',                 # 0 = all cores
           '--seed', '42', '--no-batch', '--quiet',
           '--output', str(run_dir)]
    if warm:
        cmd += ['--ip-warmstart', warm]

    print(f'\n=== IP on {ds_name} '
          f'(warm-start: {"yes" if warm else "no"}) ===', flush=True)
    try:
        subprocess.run(cmd, timeout=IP_TIME + 180, check=False)
    except subprocess.TimeoutExpired:
        with open(ip_fail, 'a') as f:
            f.write(f'{ds_name}: TIMEOUT {IP_TIME + 180}s\n\n')
        print(f'  TIMEOUT', flush=True)
    print(f'  total elapsed: {(time.time()-t0)/60:.1f}m', flush=True)

print(f'\nIP batch complete in {(time.time()-t0)/60:.1f}m.')

## 8. Zip + download IP results

Separate zip so you can choose to download just this piece. If Drive is mounted, it also lands there automatically.

In [ ]:
import os, shutil
from google.colab import files

!zip -r batch_colab_ip.zip results/colab_batch_ip/ > /dev/null

drive_dir = os.environ.get('DRIVE_BACKUP_DIR', '')
if drive_dir and os.path.isdir(drive_dir):
    shutil.copy('batch_colab_ip.zip', drive_dir)
    print(f'Drive copy: {drive_dir}/batch_colab_ip.zip')

files.download('batch_colab_ip.zip')

## 9. Scaling batch — dense synthetic ladder (100 → 1000 exams)

Runs every algorithm on a 10-step synthetic ladder so the runtime-vs-size curve has even log spacing and tight error bars. The ladder files (`instances/synthetic_scaling_{100..1000}.exam`) are regenerated in-place with seed 42 if missing, so this cell is idempotent regardless of whether the files are committed.

- **Cost**: 13 algos × 10 sizes × `SCALING_SEEDS` seeds (default 2) = ~260 runs. On A100 parallel: ~10–20 min. On T4: ~25–40 min.
- **Output**: `results/colab_batch_scaling/` with per-run dirs + an aggregated CSV keyed by `num_exams` for `plot_scaling`.

In [ ]:
import os, json, time, subprocess
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed
from core.generator import generate_synthetic, write_itc2007_format

SCALING_SIZES = list(range(100, 1001, 100))  # 100, 200, ..., 1000
SCALING_ALGOS = ['greedy', 'tabu', 'kempe', 'sa', 'alns', 'gd', 'abc', 'ga',
                 'lahc', 'woa', 'hho', 'cpsat', 'vns']
SCALING_SEEDS = 2

# Generate ladder instances if missing (idempotent — fixed seed).
for n in SCALING_SIZES:
    path = Path(f'instances/synthetic_scaling_{n}.exam')
    if not path.exists():
        p = generate_synthetic(num_exams=n, preset='competition', seed=42)
        write_itc2007_format(p, str(path))
        print(f'  generated {path}')

SCALING_ROOT = Path('results/colab_batch_scaling')
SCALING_ROOT.mkdir(parents=True, exist_ok=True)
SCALE_FAIL = SCALING_ROOT / 'failures.log'
if SCALE_FAIL.exists(): SCALE_FAIL.unlink()


def run_scaling(algo, size, seed):
    ds = f'instances/synthetic_scaling_{size}.exam'
    run_dir = SCALING_ROOT / f'{algo}_n{size}_seed{seed}'
    run_dir.mkdir(exist_ok=True)
    cmd = ['python', 'main.py', '--dataset', ds, '--algo', algo,
           '--seed', str(seed), '--no-batch', '--quiet',
           '--output', str(run_dir)]
    try:
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=900)
        return (algo, size, seed,
                'ok' if r.returncode == 0 else 'fail',
                r.stderr[-400:] if r.returncode else '')
    except subprocess.TimeoutExpired:
        return (algo, size, seed, 'timeout', '900s')


jobs = [(a, n, s) for a in SCALING_ALGOS for n in SCALING_SIZES
        for s in range(42, 42 + SCALING_SEEDS)]
MAX_WORKERS = max(1, (os.cpu_count() or 2) - 1)
print(f'Scaling: {len(SCALING_ALGOS)} algos × {len(SCALING_SIZES)} sizes × '
      f'{SCALING_SEEDS} seeds → {len(jobs)} runs across {MAX_WORKERS} workers')

started = time.time(); done = 0
with ProcessPoolExecutor(max_workers=MAX_WORKERS) as pool:
    futs = {pool.submit(run_scaling, *j): j for j in jobs}
    for fut in as_completed(futs):
        algo, size, seed, status, msg = fut.result()
        done += 1
        elapsed = time.time() - started
        eta = elapsed / done * (len(jobs) - done)
        tag = '  ok' if status == 'ok' else f' {status.upper()}'
        print(f'  [{done:>3}/{len(jobs)}] {algo:<6} n={size:<5} seed={seed}'
              f'  elapsed={elapsed/60:5.1f}m eta={eta/60:5.1f}m {tag}', flush=True)
        if status != 'ok':
            with open(SCALE_FAIL, 'a') as f:
                f.write(f'{algo}/n={size}/seed{seed}: {status}\n{msg}\n\n')

# Aggregate into a single CSV keyed by num_exams.
import pandas as pd
rows = []
for run_dir in sorted(SCALING_ROOT.iterdir()):
    if not run_dir.is_dir(): continue
    summ = run_dir / 'summary.json'
    if not summ.exists(): continue
    head, _, seed_tag = run_dir.name.rpartition('_seed')
    algo, _, n_tag = head.partition('_n')
    n_exams = int(n_tag)
    for algo_label, rec in json.loads(summ.read_text()).items():
        rows.append({
            'algorithm': algo_label, 'num_exams': n_exams,
            'seed': int(seed_tag),
            'soft_penalty': rec['soft'], 'runtime': rec['runtime'],
            'feasible': rec['feasible'], 'hard': rec['hard'],
        })
df = pd.DataFrame(rows)
df.to_csv(SCALING_ROOT / 'aggregated.csv', index=False)
print(f'\nScaling batch complete: {len(df)} rows → {SCALING_ROOT/"aggregated.csv"}')

## 10. Parameter sweep — 1-D sensitivity on 1000-exam instance

For every algo with a declared search space in `tooling/tuner/search_spaces.py`, pin every knob at its tuned default and vary one knob at a time across `SWEEP_VALUES` points (min / mid / max by default). Runs on the largest synthetic instance so soft-penalty variance doesn't drown in noise.

- **Skipped**: `greedy` (no knobs), `cpsat` / `ip` (time-budget knob only — different axis), `hho` (not in SEARCH_SPACES).
- **Cost** (10 algos, ~18 knob-axes total × 3 values × `SWEEP_SEEDS` seeds): ~160 runs at `SWEEP_SEEDS=3`. On parallel: ~8–15 min.
- **Output**: `results/colab_batch_sweep/sensitivity.csv` — consumed by `utils.plots.tuning.plot_parameter_sensitivity`.

In [ ]:
SWEEP_DATASET = 'instances/synthetic_scaling_1000.exam'
SWEEP_SEEDS = 3
SWEEP_VALUES = 3

!python -m tooling.param_sweep \
    --dataset {SWEEP_DATASET} \
    --seeds {SWEEP_SEEDS} \
    --values {SWEEP_VALUES} \
    --out results/colab_batch_sweep/sensitivity.csv \
    --runs-dir results/colab_batch_sweep/runs

## 11. Zip + download scaling + sweep

Third (and last) zip — bundles both the scaling ladder results and the sensitivity CSV. As with the other zips, if Drive is mounted it's copied there too.

In [ ]:
import os, shutil
from google.colab import files

!zip -r batch_colab_scaling_sweep.zip results/colab_batch_scaling/ results/colab_batch_sweep/ > /dev/null

drive_dir = os.environ.get('DRIVE_BACKUP_DIR', '')
if drive_dir and os.path.isdir(drive_dir):
    shutil.copy('batch_colab_scaling_sweep.zip', drive_dir)
    print(f'Drive copy: {drive_dir}/batch_colab_scaling_sweep.zip')

files.download('batch_colab_scaling_sweep.zip')